<a href="https://colab.research.google.com/github/Amazeble/chatterbox-finetuning/blob/main/colab_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook guides you through fine-tuning the Chatterbox TTS model.

## Clone Repository & Connect Gdrive

In [ ]:
#@title 🎛️ Configure Hyperparameters & Environment
#@markdown Modify the parameters below and run this cell to update `/content/chatterbox-finetuning/src/config.py`.
!git clone https://github.com/Amazeble/chatterbox-finetuning.git
# --- Colab Form Fields ---
batch_size = 32 #@param {type:"integer"}
grad_accum = 8 #@param {type:"integer"}
learning_rate = 0.0001 #@param {type:"number"}
num_epochs = 10 #@param {type:"integer"}
save_steps = 35 #@param {type:"integer"}
save_total_limit = 2 #@param {type:"integer"}
dataloader_num_workers = 2 #@param {type:"integer"}
project_name = "Seraphina" #@param {type:"string"}

#@markdown 📁 Google Drive Options
mount_drive = True #@param {type:"boolean"}
is_turbo = True #@param {type:"boolean"}
is_lora = True #@param {type:"boolean"}
is_merge_lora = False #@param {type:"boolean"}
model_dir = "./pretrained_models" #@param {type:"string"}
base_dataset_dir = "/content/drive/MyDrive/Chatterbox/MyTTSDataset" #@param {type:"string"}
base_output_dir = "/content/drive/MyDrive/Chatterbox/chatterbox_output" #@param {type:"string"}

import os
import pathlib
import re

# 1. Handle Google Drive Mount
if mount_drive:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("⏭️ Skipping Google Drive mount.")

# 2. Map form variables to their replacement patterns
updates = {
    "batch_size": f"batch_size: int = {batch_size}",
    "grad_accum": f"grad_accum: int = {grad_accum}",
    "learning_rate": f"learning_rate: float = {learning_rate}",
    "num_epochs": f"num_epochs: int = {num_epochs}",
    "save_steps": f"save_steps: int = {save_steps}",
    "save_total_limit": f"save_total_limit: int = {save_total_limit}",
    "dataloader_num_workers": f"dataloader_num_workers: int = {dataloader_num_workers}",
    "project_name": f'project_name: Optional[str] = "{project_name}"',
    "is_turbo": f"is_turbo: bool = {is_turbo}",
    "is_lora": f"is_lora: bool = {is_lora}",
    "is_merge_lora": f"is_merge_lora: bool = {is_merge_lora}",
    "model_dir": f'model_dir: str = "{model_dir}"',
    "base_dataset_dir": f'base_dataset_dir: str = "{base_dataset_dir}"',
    "base_output_dir": f'base_output_dir: str = "{base_output_dir}"',
}

config_path = pathlib.Path("/content/chatterbox-finetuning/src/config.py")

# 3. Modify the file line-by-line safely
if config_path.exists():
    lines = config_path.read_text().splitlines()
    new_lines = []

    for line in lines:
        matched = False
        # Regex checks for the exact variable name, keeping any indentation and type hints intact
        for key, new_value in updates.items():
            if re.match(r"^\s*" + re.escape(key) + r"(\s*:\s*[^=]+)?\s*=", line):
                # Calculate leading indentation
                indent = line[:len(line) - len(line.lstrip())]

                # Extract and preserve existing inline comments if present
                comment_match = re.search(r"\s*(#.*)$", line)
                comment = comment_match.group(1) if comment_match else ""

                if comment:
                    new_lines.append(f"{indent}{new_value}  {comment}")
                else:
                    new_lines.append(f"{indent}{new_value}")

                matched = True
                break

        if not matched:
            new_lines.append(line)

    # Write back the preserved and targeted contents
    config_path.write_text("\n".join(new_lines) + "\n")
    print(f"✅ Successfully updated targeted parameters in: {config_path}")
else:
    print(f"❌ Error: {config_path} does not exist. Ensure your clone cell ran successfully.")

#Install Dependancy

In [ ]:
!add-apt-repository ppa:ubuntuhandbook1/ffmpeg8
!apt-get update -qq
!apt-get install -y ffmpeg
!pip install transformers conformer s3tokenizer spacy-pkuseg pykakasi resemble-perth jedi>=0.16 accelerate peft silero-vad librosa soundfile num2words ffmpeg-python tqdm pandas safetensors tensorboard omegaconf hf_transfer pyloudnorm gdown
!pip install torchcodec==0.11 xformers --index-url https://download.pytorch.org/whl/cu128
!pip install chatterbox-tts==0.1.7 --no-deps
!pip uninstall torchao -y!pip install --pre torchao --index-url https://download.pytorch.org/whl/nightly/cu128
!pip install --pre mslk --index-url https://download.pytorch.org/whl/nightly/cu128

# RUN SETUP

In [ ]:
%cd chatterbox-finetuning/
import os
if os.path.exists("pretrained_models"):
    import shutil
    shutil.rmtree("pretrained_models")
    print("🗑️ Cleaned old pretrained_models directory")

# Run setup.py
print("\n⏳ Downloading and preparing models...\n")
!python setup.py

print("\n✅ Setup complete! Check the output above for the new_vocab_size value.")

## Start Training

In [ ]:
%cd /content/chatterbox-finetuning/
#@title 🚀 Start Training
#@markdown Check this box to resume training from a checkpoint
resume_training = True #@param {type:"boolean"}
checkpoint_path = "/content/drive/MyDrive/Chatterbox/chatterbox_output/Seraphina/checkpoint-150" #@param {type:"string"}

#@markdown Preprocessing Options
preprocess_force = False #@param {type:"boolean"}
preprocess_continue = True #@param {type:"boolean"}

# Build command arguments
cmd_args = ""
if resume_training and checkpoint_path:
    cmd_args += f" --resume \"{checkpoint_path}\""

if resume_training and not checkpoint_path:
    print("⚠️ Resume checked but no checkpoint path provided. Running normal training...")

if preprocess_force:
    cmd_args += " --preprocess-force"

if preprocess_continue:
    cmd_args += " --preprocess-continue"

!python train.py{cmd_args}

In [ ]:
#@title 📦 Merge LoRA Adapter into Base Model
#@markdown If you used LoRA training and are satisfied with the results, merge the adapter into a standalone model file.

if is_lora:
    print("Merging LoRA adapter into base model...")
    print("This creates a single .safetensors file ready for deployment.\n")

    !python merge_lora.py

    print("\n✅ Merge complete!")
    merged_model = f"chatterbox_output/{project_name}/t3_turbo_finetuned_merged.safetensors" if is_turbo else f"chatterbox_output/{project_name}/t3_finetuned_merged.safetensors"
    print(f"Merged model saved to: {merged_model}")

    if os.path.exists(merged_model):
        size_mb = os.path.getsize(merged_model) / (1024 * 1024)
        print(f"File size: {size_mb:.1f} MB")
else:
    print("ℹ️ Skipping merge - you used full fine-tuning, not LoRA")

In [ ]:
#@title 💾 Download Trained Model
#@markdown Download your trained model to Google Drive or local storage.

from google.colab import drive
import os
import shutil

# Mount Google Drive
drive.mount('/content/drive', force_remount=False)

# Determine what to save
# Use config to get the correct output_dir
from src.config import TrainConfig
cfg = TrainConfig()
output_dir = cfg.output_dir
project_name = cfg.project_name

if os.path.exists(output_dir):
    # Create backup directory in Drive
    drive_backup = f"/content/drive/MyDrive/chatterbox_models/{project_name}"
    os.makedirs(drive_backup, exist_ok=True)

    # Copy all output files
    for item in os.listdir(output_dir):
        src = os.path.join(output_dir, item)
        dst = os.path.join(drive_backup, item)

        if os.path.isfile(src):
            shutil.copy2(src, dst)
            print(f"✅ Copied: {item} ({os.path.getsize(src) / (1024*1024):.1f} MB)")
        elif os.path.isdir(src):
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print(f"✅ Copied directory: {item}/")

    print(f"\n📁 All files saved to: {drive_backup}")
    print("\nYou can also download individual files using the file browser on the left.")
else:
    print(f"❌ Output directory not found: {output_dir}")
    print("Make sure training has completed successfully.")